In [ ]:
# Example 1: Public API surface — Depends, DependencyGraph are the primary exports.
from taskiq_dependencies import Depends, DependencyGraph

print("Depends:", Depends)
print("DependencyGraph:", DependencyGraph)

# Depends is a marker class used as a default argument value.
marker = Depends(lambda: 42)
print("marker type:", type(marker).__name__)
print("marker has dependency attr:", hasattr(marker, 'dependency'))

assert callable(Depends)
assert callable(DependencyGraph)


In [ ]:
# Example 2: Build a DependencyGraph from a simple target function and resolve a sync context.
from taskiq_dependencies import Depends, DependencyGraph

def get_number() -> int:
    return 7

def target(n: int = Depends(get_number)) -> int:
    return n * 2

graph = DependencyGraph(target=target)
with graph.sync_ctx() as ctx:
    kwargs = ctx.resolve_kwargs()

print("resolved kwargs:", kwargs)
result = target(**kwargs)
print("target result:", result)
assert result == 14


In [ ]:
# Example 3: Nested dependencies — Depends inside Depends, resolved transitively.
from taskiq_dependencies import Depends, DependencyGraph

def base_value() -> int:
    return 10

def doubled(b: int = Depends(base_value)) -> int:
    return b * 2

def target(d: int = Depends(doubled)) -> int:
    return d + 1

graph = DependencyGraph(target=target)
with graph.sync_ctx() as ctx:
    kwargs = ctx.resolve_kwargs()

print("nested kwargs:", kwargs)
assert target(**kwargs) == 21


In [ ]:
# Example 4: Generator dependencies — teardown is invoked on context exit.
from typing import Iterator
from taskiq_dependencies import Depends, DependencyGraph

events = []

def resource() -> Iterator[str]:
    events.append("setup")
    yield "resource-handle"
    events.append("teardown")

def target(r: str = Depends(resource)) -> str:
    return r.upper()

graph = DependencyGraph(target=target)
with graph.sync_ctx() as ctx:
    kwargs = ctx.resolve_kwargs()
    print("inside ctx, events:", events)
    print("target output:", target(**kwargs))

print("after ctx, events:", events)
assert events == ["setup", "teardown"]


In [ ]:
# Example 5: Async dependency resolution via async_ctx.
from taskiq_dependencies import Depends, DependencyGraph

async def async_value() -> int:
    return 21

def target(v: int = Depends(async_value)) -> int:
    return v * 2

graph = DependencyGraph(target=target)
async with graph.async_ctx() as ctx:
    kwargs = await ctx.resolve_kwargs()

print("async-resolved kwargs:", kwargs)
result = target(**kwargs)
print("result:", result)
assert result == 42
